# Orders analytics

Pyodide tour over `Data/Orders.dataset`. Before each **Run**, Lattice mounts
the Orders source CSV into the Pyodide filesystem at:

`/home/pyodide/workspace/Data/Orders.dataset/sources/orders.csv`

**Native desktop** with an open workspace is required for that bridge. The
browser fixture shows an honest banner and cannot see workspace files.

DuckDB / Parquet stay on the **native** path (CLI or dataset Profile) — this
notebook does **not** load DuckDB into Pyodide. Example for a terminal in the
workspace root:

```sql
SELECT region, SUM(revenue) AS revenue
FROM read_parquet('Data/Orders.dataset/facts/**/*.parquet')
GROUP BY 1
ORDER BY revenue DESC;
```

Or: `lattice dataset query Data/Orders.dataset --sql "…"` when using the CLI.


In [ ]:
import pandas as pd

ORDERS_CSV = "/home/pyodide/workspace/Data/Orders.dataset/sources/orders.csv"

orders = pd.read_csv(ORDERS_CSV)
print("shape:", orders.shape)
print(orders.head())


## Revenue by region

Aggregate in pandas and print a clear table. Matplotlib draws a bar chart to a
PNG buffer for a textual confirmation of the figure size (Notebook outputs here
are stream/`text/plain` — use the printed table as the durable signal).


In [ ]:
import io
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

by_region = (
    orders.groupby("region", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)
print("Revenue by region")
print(by_region.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_region["region"], by_region["revenue"], color="#3d5a40")
ax.set_title("Orders revenue by region")
ax.set_ylabel("revenue")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png")
plt.close(fig)
print(f"matplotlib PNG bytes: {buf.getbuffer().nbytes}")
